In [48]:
import jax.numpy as jnp
import jax
import gymnax
from flax import nnx
import optax
from typing import Any

In [ ]:
class MLP(nnx.Module):
    def __init__(self, din: int, dout: int, *, rngs: nnx.Rngs):
        self.layer1 = nnx.Linear(din, 25, rngs=rngs)
        self.layer2 = nnx.Linear(25, 25, rngs=rngs)
        self.layer3 = nnx.Linear(25, dout, rngs=rngs)

    def __call__(self, X: jnp.ndarray) -> jnp.ndarray:
        X = nnx.relu(self.layer1(X))
        X = nnx.relu(self.layer2(X))
        X = self.layer3(X)
        return X
    
Rngs = nnx.Rngs(0, env=0, policy=0)
model = MLP(4, 2, rngs=Rngs) # 4 states in, 2 actions out [Q(s, left), Q(s, right)]
optimiser = nnx.Optimizer(model, optax.adam(1e-3), wrt=nnx.Param)
cpu_device = jax.devices('cpu')[0]
gpu_device = jax.devices('gpu')[0]

from dataclasses import replace

env, env_params = gymnax.make("CartPole-v1")
env_params = replace(env_params, max_steps_in_episode=500)
gamma = 1

scale_vector = jnp.ones(4)

def loss_fn(model: nnx.Module, obs, action, target):
    q_value: jnp.ndarray = model(obs * scale_vector)[action]
    loss = 0.5 * (target - q_value) ** 2
    return loss

@nnx.scan(in_axes=(nnx.Carry, 0), out_axes=(nnx.Carry, 0))
def policy_step(state_input: tuple[Any, Any, MLP, Any, nnx.Rngs], epsilon):
    print("compile policy step")
    obs, state, model, optimiser, rngs = state_input

    q_values: jnp.ndarray = model(obs * scale_vector)

    should_explore = rngs.policy.uniform()
    greedy_action = q_values.argmax()
    random_action = rngs.randint((), 0, 2)
    action = jnp.where(should_explore > epsilon, greedy_action, random_action)

    next_obs, next_state, reward, done, _ = env.step(
        rngs.env(), state, action, env_params
    )
    # jax.debug.print("reward={0}", reward)

    target = reward + (1-done) * gamma * jax.lax.stop_gradient(model(next_obs * scale_vector).max())

    grads = nnx.grad(loss_fn)(model, obs, action, target)
    optimiser.update(model, grads)

    return (next_obs, next_state, model, optimiser, rngs), (reward, done, state)

@nnx.scan(in_axes=(nnx.Carry, 0), out_axes=(nnx.Carry, 0))
def train_episode(carry, episode_idx):
    model, optimiser, rngs = carry
    obs, env_state = env.reset(rngs.env(), env_params)
    epsilon = jnp.maximum(0.05, 1.0 - (episode_idx / 1000.0))

    step_carry = (obs, env_state, model, optimiser, rngs)

    final_step_carry, (rewards, dones, _) = policy_step(
        step_carry, 
        jnp.full((env_params.max_steps_in_episode,), epsilon) 
    )

    _, _, final_model, final_optimiser, final_rngs = final_step_carry

    episode_reward = jnp.sum(rewards)
    first_done_idx = jnp.argmax(dones)

    did_fail = jnp.any(dones)
    
    # true Score: If it failed, use the index. If never failed, use 500
    true_score = jnp.where(did_fail, first_done_idx + 1.0, env_params.max_steps_in_episode)
    
    # it is solved if it balanced for the full duration
    solved = true_score >= env_params.max_steps_in_episode

    new_carry = (final_model, final_optimiser, final_rngs)

    return new_carry, (true_score, solved, epsilon)

# train_episode_cpu = jax.jit(train_episode, device=cpu_device)

final, (rewards, solved_mask, epsilons) = train_episode((model, optimiser, Rngs), jnp.arange(1000))
# jax.debug.print("Episode finished. Final Position: {0}, Epsilon: {1}, Episode: {2}", final[0][0], epsilon, episode_idx)



compile policy step


In [50]:
size = 50
reshaped_rewards = rewards.reshape(-1, size).mean(axis=1)
for i, r in enumerate(reshaped_rewards):
    print(f"Ep {i*size} - {(i+1)*size}: Avg Reward {r:.2f} | Solved: {jnp.sum(solved_mask.reshape(-1, size)[i])} times")

Ep 0 - 50: Avg Reward 22.68 | Solved: 0 times
Ep 50 - 100: Avg Reward 25.54 | Solved: 0 times
Ep 100 - 150: Avg Reward 24.78 | Solved: 0 times
Ep 150 - 200: Avg Reward 35.54 | Solved: 0 times
Ep 200 - 250: Avg Reward 41.72 | Solved: 0 times
Ep 250 - 300: Avg Reward 53.84 | Solved: 0 times
Ep 300 - 350: Avg Reward 56.30 | Solved: 0 times
Ep 350 - 400: Avg Reward 85.80 | Solved: 0 times
Ep 400 - 450: Avg Reward 83.98 | Solved: 0 times
Ep 450 - 500: Avg Reward 139.22 | Solved: 0 times
Ep 500 - 550: Avg Reward 211.50 | Solved: 5 times
Ep 550 - 600: Avg Reward 279.28 | Solved: 8 times
Ep 600 - 650: Avg Reward 320.24 | Solved: 19 times
Ep 650 - 700: Avg Reward 279.52 | Solved: 11 times
Ep 700 - 750: Avg Reward 330.34 | Solved: 6 times
Ep 750 - 800: Avg Reward 398.82 | Solved: 17 times
Ep 800 - 850: Avg Reward 496.00 | Solved: 49 times
Ep 850 - 900: Avg Reward 499.08 | Solved: 49 times
Ep 900 - 950: Avg Reward 495.74 | Solved: 46 times
Ep 950 - 1000: Avg Reward 499.76 | Solved: 48 times


In [51]:
obs, env_state = env.reset(Rngs.env(), env_params)
epsilon = jnp.array(0.0)

step_carry = (obs, env_state, model, optimiser, Rngs)

final_step_carry, (rewards, dones, states) = policy_step(
    step_carry, 
    jnp.full((env_params.max_steps_in_episode,), epsilon) 
)

batch_size = states.time.shape[0]
state_list = [
    jax.tree_util.tree_map(lambda x: x[i], states) 
    for i in range(batch_size)
]

In [ ]:
import gymnasium as gym 
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

def save_cartpole_gif(state_list, filename="cartpole_run.gif"):
    print(f"Generating animation ({len(state_list)} frames)...")
    
    env = gym.make("CartPole-v1", render_mode="rgb_array")
    
    frames = []
    fig = plt.figure(figsize=(6, 4))
    plt.axis('off')
    
    for state in state_list:
        env.reset()
        
        gym_state = np.array([
            float(state.x), 
            float(state.x_dot), 
            float(state.theta), 
            float(state.theta_dot)
        ])
        
        env.unwrapped.state = gym_state

        frame = env.render()
        
        im = plt.imshow(frame, animated=True)
        frames.append([im])

    ani = animation.ArtistAnimation(fig, frames, interval=50, blit=True)
    ani.save(filename, writer='pillow')
    print(f"Saved to {filename}")
    plt.close()

save_cartpole_gif(state_list, "my_cartpole.gif")

Generating animation (500 frames)...
Saved to my_cartpole.gif
